In [2]:
!pip -q install opencv-contrib-python reportlab matplotlib scipy pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.3 MB/s eta 0:00:00


In [7]:
!pip -q install opencv-contrib-python reportlab matplotlib scipy pillow

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial import cKDTree
from google.colab import files

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Image as RLImage,
    Table, TableStyle, PageBreak
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet


def matrix_to_string(M, precision=5):
    return np.array2string(M, precision=precision, suppress_small=False)


def read_image(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    return img


def save_rgb(path, bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    Image.fromarray(rgb).save(path)


def draw_inlier_matches(img1, kp1, img2, kp2, matches, mask, max_draw=80):
    inlier_matches = []
    if mask is not None:
        mask = mask.ravel().astype(bool)
        for m, keep in zip(matches, mask):
            if keep:
                inlier_matches.append(m)
    else:
        inlier_matches = matches

    inlier_matches = sorted(inlier_matches, key=lambda m: m.distance)[:max_draw]
    vis = cv2.drawMatches(
        img1, kp1, img2, kp2, inlier_matches, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    return vis


def annotate_point_and_text(img, point, text_lines, color=(0, 0, 255)):
    out = img.copy()
    x, y = int(point[0]), int(point[1])
    cv2.circle(out, (x, y), 10, color, -1)
    cv2.circle(out, (x, y), 18, color, 2)

    y0 = max(30, y - 20)
    for i, line in enumerate(text_lines):
        yy = y0 + i * 30
        cv2.putText(
            out, line, (x + 20, yy),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA
        )
    return out


def compute_camera_center_distance(X_world_h, C_world_h):
    return np.linalg.norm(X_world_h[:3] - C_world_h[:3])


def triangulate_point(P1, P2, pt1, pt2):
    p1 = np.array(pt1, dtype=float).reshape(2, 1)
    p2 = np.array(pt2, dtype=float).reshape(2, 1)
    X_h = cv2.triangulatePoints(P1, P2, p1, p2)
    X_h = (X_h / X_h[3]).reshape(-1)
    return X_h


def reproject_point(P, X_h):
    x = P @ X_h.reshape(4, 1)
    x = x / x[2]
    return x[:2].ravel()


def create_feature_detector():
    if hasattr(cv2, "SIFT_create"):
        detector = cv2.SIFT_create(nfeatures=4000)
        norm_type = cv2.NORM_L2
        method = "SIFT"
    else:
        detector = cv2.ORB_create(nfeatures=5000)
        norm_type = cv2.NORM_HAMMING
        method = "ORB"
    return detector, norm_type, method


def match_features(img1, img2):
    detector, norm_type, method = create_feature_detector()

    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

    kp1, des1 = detector.detectAndCompute(gray1, None)
    kp2, des2 = detector.detectAndCompute(gray2, None)

    if des1 is None or des2 is None:
        raise ValueError("Feature detection failed.")

    bf = cv2.BFMatcher(norm_type)
    raw = bf.knnMatch(des1, des2, k=2)

    good = []
    ratio = 0.75 if method == "SIFT" else 0.8
    for m, n in raw:
        if m.distance < ratio * n.distance:
            good.append(m)

    if len(good) < 8:
        raise ValueError("Not enough good matches for F estimation.")

    pts1 = np.float32([kp1[m.queryIdx].pt for m in good])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in good])

    return kp1, kp2, good, pts1, pts2, method


def estimate_fundamental(pts1, pts2):
    F, mask = cv2.findFundamentalMat(
        pts1, pts2,
        method=cv2.FM_RANSAC,
        ransacReprojThreshold=1.0,
        confidence=0.999
    )
    if F is None or F.shape != (3, 3):
        raise ValueError("Fundamental matrix estimation failed.")
    return F, mask


def estimate_essential(F, K):
    E = K.T @ F @ K
    U, S, Vt = np.linalg.svd(E)
    S = np.array([1.0, 1.0, 0.0])
    E = U @ np.diag(S) @ Vt
    return E


def recover_pose_from_E(E, pts1, pts2, K):
    retval, R, t, pose_mask = cv2.recoverPose(E, pts1, pts2, K)
    return retval, R, t, pose_mask


def get_default_intrinsics(w, h):
    fx = 0.9 * max(w, h)
    fy = 0.9 * max(w, h)
    cx = w / 2.0
    cy = h / 2.0
    return np.array([
        [fx, 0,  cx],
        [0,  fy, cy],
        [0,  0,  1 ]
    ], dtype=float)


def nearest_match_to_clicked_point(clicked_pt, pts1, pts2, mask):
    mask = mask.ravel().astype(bool)
    in1 = pts1[mask]
    in2 = pts2[mask]
    tree = cKDTree(in1)
    dist, idx = tree.query(np.array(clicked_pt), k=1)
    return in1[idx], in2[idx], dist


def display_and_click(image_bgr, title="Click a point"):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 10))
    plt.imshow(rgb)
    plt.title(title)
    plt.axis("on")
    plt.show()

    print(title)
    x = float(input("Enter x pixel coordinate: ").strip())
    y = float(input("Enter y pixel coordinate: ").strip())
    return (x, y)


def make_report(
    output_pdf,
    student_name,
    course_name,
    object_name,
    baseline_m,
    ground_truth_m,
    est_dist_cam1_m,
    est_dist_cam2_m,
    abs_error_m,
    rel_error_pct,
    K,
    F,
    E,
    R,
    t,
    notes,
    left_img_path,
    right_img_path,
    matches_img_path,
    annotated_left_path,
    annotated_setup_path,
):
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("Uncalibrated Stereo Distance Estimation Report", styles["Title"]))
    story.append(Spacer(1, 0.2 * inch))

    intro = f"""
    <b>Student:</b> {student_name}<br/>
    <b>Course:</b> {course_name}<br/>
    <b>Selected Object:</b> {object_name}<br/>
    <b>Measured Baseline:</b> {baseline_m:.4f} m<br/>
    <b>Ground Truth Distance:</b> {ground_truth_m:.4f} m
    """
    story.append(Paragraph(intro, styles["BodyText"]))
    story.append(Spacer(1, 0.2 * inch))

    methodology = """
    <b>Methodology</b><br/>
    Two images of the same object were taken from slightly different viewpoints.
    Feature correspondences were extracted between the left and right images, and
    the Fundamental Matrix (F) was estimated using RANSAC. An approximate intrinsic
    matrix K was used to compute the Essential Matrix (E = K^T F K). Relative pose
    was recovered from E, giving the Rotation Matrix (R) and translation direction t.
    The selected object point was triangulated into 3D, and metric scale was introduced
    using the measured stereo baseline.
    """
    story.append(Paragraph(methodology, styles["BodyText"]))
    story.append(Spacer(1, 0.2 * inch))

    results_data = [
        ["Quantity", "Value"],
        ["Estimated distance from left camera", f"{est_dist_cam1_m:.4f} m"],
        ["Estimated distance from right camera", f"{est_dist_cam2_m:.4f} m"],
        ["Ground truth distance", f"{ground_truth_m:.4f} m"],
        ["Absolute error", f"{abs_error_m:.4f} m"],
        ["Relative error", f"{rel_error_pct:.2f} %"],
    ]
    table = Table(results_data, colWidths=[3.2 * inch, 2.3 * inch])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 1, colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ]))
    story.append(Paragraph("<b>Results</b>", styles["Heading2"]))
    story.append(table)
    story.append(Spacer(1, 0.2 * inch))

    story.append(Paragraph("<b>Intrinsic Matrix (K)</b>", styles["Heading2"]))
    story.append(Paragraph(f"<font name='Courier'>{matrix_to_string(K)}</font>", styles["Code"]))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Fundamental Matrix (F)</b>", styles["Heading2"]))
    story.append(Paragraph(f"<font name='Courier'>{matrix_to_string(F)}</font>", styles["Code"]))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Essential Matrix (E)</b>", styles["Heading2"]))
    story.append(Paragraph(f"<font name='Courier'>{matrix_to_string(E)}</font>", styles["Code"]))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Rotation Matrix (R)</b>", styles["Heading2"]))
    story.append(Paragraph(f"<font name='Courier'>{matrix_to_string(R)}</font>", styles["Code"]))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Translation Vector (scaled)</b>", styles["Heading2"]))
    story.append(Paragraph(f"<font name='Courier'>{matrix_to_string(t)}</font>", styles["Code"]))
    story.append(Spacer(1, 0.2 * inch))

    if notes.strip():
        story.append(Paragraph("<b>Notes</b>", styles["Heading2"]))
        story.append(Paragraph(notes.replace("\n", "<br/>"), styles["BodyText"]))
        story.append(Spacer(1, 0.2 * inch))

    story.append(PageBreak())

    story.append(Paragraph("<b>Left Image</b>", styles["Heading2"]))
    story.append(RLImage(left_img_path, width=5.7 * inch, height=7.0 * inch))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Right Image</b>", styles["Heading2"]))
    story.append(RLImage(right_img_path, width=5.7 * inch, height=7.0 * inch))
    story.append(Spacer(1, 0.15 * inch))

    story.append(PageBreak())

    story.append(Paragraph("<b>Feature Matches</b>", styles["Heading2"]))
    story.append(RLImage(matches_img_path, width=6.5 * inch, height=3.0 * inch))
    story.append(Spacer(1, 0.2 * inch))

    story.append(Paragraph("<b>Annotated Left Image</b>", styles["Heading2"]))
    story.append(RLImage(annotated_left_path, width=5.7 * inch, height=7.0 * inch))
    story.append(Spacer(1, 0.15 * inch))

    story.append(Paragraph("<b>Annotated Setup Image</b>", styles["Heading2"]))
    story.append(RLImage(annotated_setup_path, width=5.7 * inch, height=7.0 * inch))

    doc = SimpleDocTemplate(output_pdf, pagesize=A4)
    doc.build(story)


print("=" * 60)
print("UNCALIBRATED STEREO ASSIGNMENT")
print("=" * 60)

left_file = "left.jpg"
right_file = "right.jpg"
setup_file = "setup.jpg"

for f in [left_file, right_file, setup_file]:
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing file: {f}")

img_left = read_image(left_file)
img_right = read_image(right_file)
img_setup = read_image(setup_file)

h, w = img_left.shape[:2]
print(f"Image size: {w} x {h}")

student_name = input("Student name: ").strip()
course_name = input("Course name: ").strip()
object_name = input("Object name: ").strip()

baseline_m = float(input("Baseline in meters: ").strip())
ground_truth_m = float(input("Ground truth distance in meters: ").strip())

use_default_K = input("Use default intrinsics? (y/n): ").strip().lower()
if use_default_K == "y":
    K = get_default_intrinsics(w, h)
else:
    fx = float(input("fx: ").strip())
    fy = float(input("fy: ").strip())
    cx = float(input(f"cx [{w/2:.1f}]: ").strip() or (w / 2))
    cy = float(input(f"cy [{h/2:.1f}]: ").strip() or (h / 2))
    K = np.array([
        [fx, 0, cx],
        [0, fy, cy],
        [0, 0, 1]
    ], dtype=float)

print("\nK =\n", K)

kp1, kp2, matches, pts1, pts2, method = match_features(img_left, img_right)
print(f"\nFeature method: {method}")
print(f"Good matches: {len(matches)}")

F, maskF = estimate_fundamental(pts1, pts2)
print("\nF =\n", F)
print("RANSAC inliers:", int(maskF.sum()))

E = estimate_essential(F, K)
print("\nE =\n", E)

retval, R, t, pose_mask = recover_pose_from_E(E, pts1, pts2, K)
print("\nR =\n", R)
print("\nt =\n", t)

P1 = K @ np.hstack([np.eye(3), np.zeros((3, 1))])

t_unit = t / np.linalg.norm(t)
t_metric = t_unit * baseline_m
P2 = K @ np.hstack([R, t_metric.reshape(3, 1)])

print("\nPick a clear point on the bottle label in the LEFT image.")
clicked_left = display_and_click(img_left, "LEFT image")
nearest_left, nearest_right_from_match, click_dist = nearest_match_to_clicked_point(
    clicked_left, pts1, pts2, maskF
)
print(f"Nearest inlier distance from click: {click_dist:.2f} px")

use_auto_right = input("Use matched point in RIGHT image? (y/n): ").strip().lower()
if use_auto_right == "y":
    selected_left = nearest_left
    selected_right = nearest_right_from_match
else:
    print("\nPick the same point in the RIGHT image.")
    clicked_right = display_and_click(img_right, "RIGHT image")
    selected_left = np.array(clicked_left, dtype=float)
    selected_right = np.array(clicked_right, dtype=float)

print("\nSelected LEFT point:", selected_left)
print("Selected RIGHT point:", selected_right)

X_h = triangulate_point(P1, P2, selected_left, selected_right)
X_world = X_h[:3]

C1 = np.array([0, 0, 0, 1.0])
C2_xyz = -R.T @ t_metric.reshape(3, 1)
C2 = np.array([C2_xyz[0, 0], C2_xyz[1, 0], C2_xyz[2, 0], 1.0])

dist_cam1 = compute_camera_center_distance(X_h, C1)
dist_cam2 = compute_camera_center_distance(X_h, C2)

abs_error = abs(dist_cam1 - ground_truth_m)
rel_error = 100.0 * abs_error / ground_truth_m if ground_truth_m > 0 else 0.0

print("\nTriangulated 3D point:", X_world)
print(f"Estimated distance from left camera:  {dist_cam1:.4f} m")
print(f"Estimated distance from right camera: {dist_cam2:.4f} m")
print(f"Ground truth distance:                {ground_truth_m:.4f} m")
print(f"Absolute error:                       {abs_error:.4f} m")
print(f"Relative error:                       {rel_error:.2f} %")

rp1 = reproject_point(P1, X_h)
rp2 = reproject_point(P2, X_h)
err1 = np.linalg.norm(rp1 - np.array(selected_left))
err2 = np.linalg.norm(rp2 - np.array(selected_right))
print(f"Reprojection error left:              {err1:.3f} px")
print(f"Reprojection error right:             {err2:.3f} px")

matches_vis = draw_inlier_matches(img_left, kp1, img_right, kp2, matches, maskF)

annot_left = annotate_point_and_text(
    img_left,
    selected_left,
    [
        f"Object: {object_name}",
        f"Estimated distance: {dist_cam1:.3f} m",
        f"Ground truth: {ground_truth_m:.3f} m"
    ],
    color=(0, 0, 255)
)

print("\nPick the object location in the SETUP image.")
setup_pt = display_and_click(img_setup, "SETUP image")

annot_setup = annotate_point_and_text(
    img_setup,
    setup_pt,
    [
        f"Object: {object_name}",
        f"Estimated distance: {dist_cam1:.3f} m",
        f"Ground truth: {ground_truth_m:.3f} m",
        f"Baseline: {baseline_m:.3f} m"
    ],
    color=(255, 0, 0)
)

os.makedirs("stereo_report_output", exist_ok=True)

left_out = "stereo_report_output/left_image.png"
right_out = "stereo_report_output/right_image.png"
matches_out = "stereo_report_output/inlier_matches.png"
annot_left_out = "stereo_report_output/annotated_left.png"
annot_setup_out = "stereo_report_output/annotated_setup.png"

save_rgb(left_out, img_left)
save_rgb(right_out, img_right)
save_rgb(matches_out, matches_vis)
save_rgb(annot_left_out, annot_left)
save_rgb(annot_setup_out, annot_setup)

with open("stereo_report_output/matrices_and_results.txt", "w") as f:
    f.write("UNCALIBRATED STEREO ASSIGNMENT RESULTS\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Student: {student_name}\n")
    f.write(f"Course: {course_name}\n")
    f.write(f"Object: {object_name}\n")
    f.write(f"Baseline (m): {baseline_m}\n")
    f.write(f"Ground truth distance (m): {ground_truth_m}\n\n")
    f.write("K =\n" + matrix_to_string(K) + "\n\n")
    f.write("F =\n" + matrix_to_string(F) + "\n\n")
    f.write("E =\n" + matrix_to_string(E) + "\n\n")
    f.write("R =\n" + matrix_to_string(R) + "\n\n")
    f.write("t =\n" + matrix_to_string(t_metric.reshape(3, 1)) + "\n\n")
    f.write(f"Selected left point: {selected_left}\n")
    f.write(f"Selected right point: {selected_right}\n")
    f.write(f"Triangulated world point: {X_world}\n")
    f.write(f"Estimated distance from left camera (m): {dist_cam1}\n")
    f.write(f"Estimated distance from right camera (m): {dist_cam2}\n")
    f.write(f"Absolute error (m): {abs_error}\n")
    f.write(f"Relative error (%): {rel_error}\n")
    f.write(f"Reprojection error left (px): {err1}\n")
    f.write(f"Reprojection error right (px): {err2}\n")

notes = input("\nExtra notes for report (optional): ").strip()

pdf_path = "stereo_report_output/Uncalibrated_Stereo_Report.pdf"
make_report(
    output_pdf=pdf_path,
    student_name=student_name,
    course_name=course_name,
    object_name=object_name,
    baseline_m=baseline_m,
    ground_truth_m=ground_truth_m,
    est_dist_cam1_m=dist_cam1,
    est_dist_cam2_m=dist_cam2,
    abs_error_m=abs_error,
    rel_error_pct=rel_error,
    K=K,
    F=F,
    E=E,
    R=R,
    t=t_metric.reshape(3, 1),
    notes=notes,
    left_img_path=left_out,
    right_img_path=right_out,
    matches_img_path=matches_out,
    annotated_left_path=annot_left_out,
    annotated_setup_path=annot_setup_out,
)

print("\nDone.")
print("Files in stereo_report_output:")
for f in os.listdir("stereo_report_output"):
    print("-", f)

files.download(pdf_path)

UNCALIBRATED STEREO ASSIGNMENT
Image size: 960 x 1280
Student name: pooja
Course name: computer vision
Object name: spray bottle
Baseline in meters: 0.07
Ground truth distance in meters: 0.80
Use default intrinsics? (y/n): y

K =
 [[1.152e+03 0.000e+00 4.800e+02]
 [0.000e+00 1.152e+03 6.400e+02]
 [0.000e+00 0.000e+00 1.000e+00]]

Feature method: SIFT
Good matches: 438

F =
 [[ 3.57825000e-07  1.34329356e-06 -6.05711091e-03]
 [-4.22351397e-06  1.12941219e-06  4.95652897e-02]
 [ 5.52809582e-03 -5.25017430e-02  1.00000000e+00]]
RANSAC inliers: 337

E =
 [[ 8.47392388e-03  3.29610765e-02 -1.03962302e-01]
 [-9.82271474e-02 -9.08061438e-04  9.89674659e-01]
 [ 5.58885791e-02 -9.97899469e-01  1.17875840e-03]]

R =
 [[ 0.99664395 -0.04870128  0.06579537]
 [ 0.04842931  0.99881022  0.00572315]
 [-0.06599582 -0.00251752  0.99781672]]

t =
 [[-0.99399879]
 [-0.10437755]
 [-0.03273725]]

Pick a clear point on the bottle label in the LEFT image.
LEFT image
Enter x pixel coordinate: 1600
Enter y pixe

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>